In [ ]:
# Google Colab Notebook: Data Preprocessing & Feature Engineering

"""
MASKED IP DETECTION - DATA PREPROCESSING
=========================================
Feature engineering and dataset preparation
"""

# ============================================================================
# CELL 1: Setup
# ============================================================================

!pip install -q pandas numpy scikit-learn tqdm ipaddress

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tqdm.notebook import tqdm
import ipaddress
import os

project_dir = '/content/drive/MyDrive/masked_ip_detection'

# ============================================================================
# CELL 2: Load Raw Data
# ============================================================================

print("Loading raw datasets...")

# Load all collected data
tor_df = pd.read_csv(f'{project_dir}/data/raw/tor_data.csv')
proxy_df = pd.read_csv(f'{project_dir}/data/raw/proxy_data.csv')
vpn_df = pd.read_csv(f'{project_dir}/data/raw/vpn_data.csv')
dc_df = pd.read_csv(f'{project_dir}/data/raw/datacenter_data.csv')
legit_df = pd.read_csv(f'{project_dir}/data/raw/legitimate_data.csv')

print(f"Tor IPs: {len(tor_df)}")
print(f"Proxy IPs: {len(proxy_df)}")
print(f"VPN Data: {len(vpn_df)}")
print(f"Datacenter Data: {len(dc_df)}")
print(f"Legitimate Data: {len(legit_df)}")

# ============================================================================
# CELL 3: Basic Feature Extraction
# ============================================================================

def extract_basic_features(ip_str):
    """Extract basic features from IP address"""
    try:
        ip = ipaddress.ip_address(ip_str)
        
        features = {
            'ip': ip_str,
            'ip_version': ip.version,
            'is_private': int(ip.is_private),
            'is_reserved': int(ip.is_reserved),
            'is_loopback': int(ip.is_loopback),
            'is_multicast': int(ip.is_multicast),
        }
        
        # IPv4 octets
        if ip.version == 4:
            octets = ip_str.split('.')
            features['octet_1'] = int(octets[0])
            features['octet_2'] = int(octets[1])
            features['octet_3'] = int(octets[2])
            features['octet_4'] = int(octets[3])
            
            # Class of IP (A, B, C)
            first_octet = int(octets[0])
            if first_octet < 128:
                features['ip_class'] = 1  # Class A
            elif first_octet < 192:
                features['ip_class'] = 2  # Class B
            elif first_octet < 224:
                features['ip_class'] = 3  # Class C
            else:
                features['ip_class'] = 4  # Class D/E
        else:
            features['octet_1'] = 0
            features['octet_2'] = 0
            features['octet_3'] = 0
            features['octet_4'] = 0
            features['ip_class'] = 0
        
        return features
    except:
        return None

print("\nExtracting basic features from Tor IPs...")
tor_features = []
for ip in tqdm(tor_df['ip'].unique()):
    feat = extract_basic_features(ip)
    if feat:
        feat['label'] = 1
        feat['type'] = 'tor'
        feat['in_tor_list'] = 1
        feat['in_proxy_list'] = 0
        feat['in_vpn_list'] = 0
        tor_features.append(feat)

print("Extracting basic features from Proxy IPs...")
proxy_features = []
for ip in tqdm(proxy_df['ip'].unique()):
    feat = extract_basic_features(ip)
    if feat:
        feat['label'] = 1
        feat['type'] = 'proxy'
        feat['in_tor_list'] = 0
        feat['in_proxy_list'] = 1
        feat['in_vpn_list'] = 0
        proxy_features.append(feat)

# ============================================================================
# CELL 4: Generate Legitimate IP Features
# ============================================================================

print("\nGenerating legitimate IP samples...")

# Generate diverse legitimate IPs
legitimate_features = []

# Common residential IP ranges
residential_ranges = [
    # Comcast
    ('73.0.0.0', '73.255.255.255'),
    ('98.192.0.0', '98.207.255.255'),
    # AT&T
    ('99.0.0.0', '99.255.255.255'),
    ('107.0.0.0', '107.127.255.255'),
    # Verizon
    ('71.0.0.0', '71.255.255.255'),
    ('108.0.0.0', '108.255.255.255'),
]

def ip_range_to_list(start_ip, end_ip, sample_size=500):
    """Generate random IPs from range"""
    start = int(ipaddress.IPv4Address(start_ip))
    end = int(ipaddress.IPv4Address(end_ip))
    
    ips = []
    for _ in range(min(sample_size, end - start)):
        random_int = np.random.randint(start, end)
        ips.append(str(ipaddress.IPv4Address(random_int)))
    
    return ips

for start, end in residential_ranges:
    sample_ips = ip_range_to_list(start, end, 300)
    for ip in tqdm(sample_ips, desc=f"Processing {start}"):
        feat = extract_basic_features(ip)
        if feat:
            feat['label'] = 0
            feat['type'] = 'legitimate'
            feat['in_tor_list'] = 0
            feat['in_proxy_list'] = 0
            feat['in_vpn_list'] = 0
            legitimate_features.append(feat)

print(f"Generated {len(legitimate_features)} legitimate IP features")

# ============================================================================
# CELL 5: Combine and Create Dataset
# ============================================================================

# Combine all features
all_features = tor_features + proxy_features + legitimate_features

df_features = pd.DataFrame(all_features)

print(f"\nTotal samples: {len(df_features)}")
print(f"\nLabel distribution:")
print(df_features['label'].value_counts())
print(f"\nType distribution:")
print(df_features['type'].value_counts())

# ============================================================================
# CELL 6: Add Synthetic Geographic Features
# ============================================================================

print("\nAdding synthetic geographic features...")

# Assign realistic coordinates based on IP type
def assign_geo_features(row):
    if row['type'] == 'tor':
        # Tor nodes are globally distributed
        row['latitude'] = np.random.uniform(-60, 70)
        row['longitude'] = np.random.uniform(-180, 180)
        row['accuracy_radius'] = np.random.randint(50, 1000)
    elif row['type'] == 'proxy':
        # Proxies often in data centers
        row['latitude'] = np.random.choice([37.4, 51.5, 1.3, -33.9])  # Major cities
        row['longitude'] = np.random.choice([-122.1, -0.1, 103.8, 151.2])
        row['accuracy_radius'] = np.random.randint(10, 100)
    else:
        # Legitimate IPs - US focused for this example
        row['latitude'] = np.random.uniform(25, 49)  # Continental US
        row['longitude'] = np.random.uniform(-125, -66)
        row['accuracy_radius'] = np.random.randint(5, 50)
    
    return row

df_features = df_features.apply(assign_geo_features, axis=1)

# ============================================================================
# CELL 7: Add ASN and Network Features
# ============================================================================

print("\nAdding ASN and network features...")

def assign_asn(row):
    if row['type'] == 'tor':
        # Random hosting ASNs
        row['asn'] = np.random.choice([16276, 24940, 20473, 14061])
    elif row['type'] == 'proxy':
        # VPS/Hosting ASNs
        row['asn'] = np.random.choice([16509, 15169, 8075, 14061])
    else:
        # Residential ISP ASNs
        row['asn'] = np.random.choice([7922, 20115, 7018, 701])
    
    return row

df_features = df_features.apply(assign_asn, axis=1)

# Add DNS features
df_features['has_ptr_record'] = np.where(
    df_features['type'] == 'legitimate',
    np.random.choice([0, 1], size=len(df_features), p=[0.3, 0.7]),
    np.random.choice([0, 1], size=len(df_features), p=[0.7, 0.3])
)

df_features['ptr_contains_host'] = np.where(
    df_features['type'].isin(['tor', 'proxy']),
    np.random.choice([0, 1], size=len(df_features), p=[0.4, 0.6]),
    np.random.choice([0, 1], size=len(df_features), p=[0.9, 0.1])
)

# ============================================================================
# CELL 8: Add Behavioral Features
# ============================================================================

print("\nAdding behavioral features...")

# Request patterns
df_features['request_count'] = np.where(
    df_features['label'] == 1,
    np.random.randint(1, 10, size=len(df_features)),
    np.random.randint(1, 50, size=len(df_features))
)

df_features['unique_user_agents'] = np.where(
    df_features['label'] == 1,
    np.random.randint(1, 3, size=len(df_features)),
    np.random.randint(1, 5, size=len(df_features))
)

# ============================================================================
# CELL 9: Data Cleaning
# ============================================================================

print("\nCleaning data...")

# Remove any rows with missing values
df_features = df_features.dropna()

# Remove duplicates based on IP
df_features = df_features.drop_duplicates(subset=['ip'])

# Ensure all numeric columns are proper types
numeric_columns = [
    'ip_version', 'is_private', 'is_reserved', 'is_loopback', 'is_multicast',
    'octet_1', 'octet_2', 'octet_3', 'octet_4', 'ip_class',
    'latitude', 'longitude', 'accuracy_radius', 'asn',
    'has_ptr_record', 'ptr_contains_host',
    'in_tor_list', 'in_proxy_list', 'in_vpn_list',
    'request_count', 'unique_user_agents', 'label'
]

for col in numeric_columns:
    if col in df_features.columns:
        df_features[col] = pd.to_numeric(df_features[col], errors='coerce')

df_features = df_features.dropna()

print(f"Final dataset shape: {df_features.shape}")

# ============================================================================
# CELL 10: Save Processed Data
# ============================================================================

output_dir = f'{project_dir}/data/processed'
os.makedirs(output_dir, exist_ok=True)

output_file = f'{output_dir}/features_dataset.csv'
df_features.to_csv(output_file, index=False)

print(f"\n✓ Processed dataset saved to: {output_file}")

# ============================================================================
# CELL 11: Dataset Statistics
# ============================================================================

print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)

print(f"\nTotal samples: {len(df_features)}")
print(f"\nLabel distribution:")
print(df_features['label'].value_counts())
print(f"\nPercentages:")
print(df_features['label'].value_counts(normalize=True) * 100)

print(f"\nType distribution:")
print(df_features['type'].value_counts())

print(f"\nFeature columns ({len(df_features.columns)}):")
print(df_features.columns.tolist())

print(f"\nSample statistics:")
print(df_features.describe())

# ============================================================================
# CELL 12: Visualize Features
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Label distribution
df_features['label'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['green', 'red'])
axes[0, 0].set_title('Label Distribution')
axes[0, 0].set_xlabel('Label')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_xticklabels(['Legitimate', 'Masked'], rotation=0)

# 2. Type distribution
df_features['type'].value_counts().plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('IP Type Distribution')
axes[0, 1].set_xlabel('Type')
axes[0, 1].set_ylabel('Count')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Geographic distribution
masked = df_features[df_features['label'] == 1]
legit = df_features[df_features['label'] == 0]

axes[0, 2].scatter(legit['longitude'], legit['latitude'], alpha=0.3, s=1, label='Legitimate', c='green')
axes[0, 2].scatter(masked['longitude'], masked['latitude'], alpha=0.3, s=1, label='Masked', c='red')
axes[0, 2].set_title('Geographic Distribution')
axes[0, 2].set_xlabel('Longitude')
axes[0, 2].set_ylabel('Latitude')
axes[0, 2].legend()

# 4. Octet 1 distribution
axes[1, 0].hist([legit['octet_1'], masked['octet_1']], bins=30, alpha=0.6, label=['Legitimate', 'Masked'], color=['green', 'red'])
axes[1, 0].set_title('First Octet Distribution')
axes[1, 0].set_xlabel('Octet Value')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# 5. Request count comparison
axes[1, 1].boxplot([legit['request_count'], masked['request_count']], labels=['Legitimate', 'Masked'])
axes[1, 1].set_title('Request Count Distribution')
axes[1, 1].set_ylabel('Request Count')

# 6. Correlation heatmap (sample)
correlation_features = ['octet_1', 'latitude', 'longitude', 'asn', 'request_count', 'label']
corr_matrix = df_features[correlation_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 2])
axes[1, 2].set_title('Feature Correlation')

plt.tight_layout()
plt.savefig(f'{project_dir}/feature_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Preprocessing complete!")
print(f"✓ Dataset ready for training!")